In [ ]:
import torch

# si ringrazia la gentile partecipazione del server Linux per la generazione del dataset (METIS non funziona su Windows)
#dataset = RedditSubGraphDataset(path='Reddit', num_graphs=1000, task_type='a')

dataset = torch.load("reddit_task3_clusters_b.pt", weights_only=False)

In [ ]:
#raccogliamo tutti gli ID unici originali presenti nel dataset
id_originali = set()
for split in ['train', 'val', 'test']:
    for subgraph in dataset[split]:
        id_originali.add(subgraph.y.item())

#ordiniamo gli ID per avere una mappatura 
id_originali = sorted(list(id_originali))
num_classi_reali = len(id_originali)

#creiamo un dizionario di conversione
mappa_id = {id_vecchio: id_nuovo for id_nuovo, id_vecchio in enumerate(id_originali)}

print(f"Community uniche trovate: {num_classi_reali}")

# applichiamo la rimappatura a tutti i sotto-grafi del dataset
for split in ['train', 'val', 'test']:
    for subgraph in dataset[split]:
        id_vecchio = subgraph.y.item()
        id_nuovo = mappa_id[id_vecchio]
        
        #sovrascriviamo il target con il nuovo indice compresso (da 0 a 39)
        subgraph.y = torch.tensor(id_nuovo, dtype=torch.long)

print("\n[OK] Rimappatura completata!")

Community uniche trovate: 39
Esempio di mappatura: ID originale 0 diventa -> 0

[OK] Rimappatura completata! Ora puoi usare out_channels = 39


In [3]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(dataset["train"], batch_size=128, shuffle=True)
val_loader = DataLoader(dataset["val"], batch_size=128, shuffle=False)
test_loader = DataLoader(dataset["test"], batch_size=128, shuffle=False)

In [ ]:
import numpy as np
from collections import Counter
from utils_task3_b import train_loop
from utils_task3_b import GCNClassifier,evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lr = 2e-4
epochs = 50

#estraiamo tutte le etichette dal train set
train_labels = [subgraph.y.item() for subgraph in dataset['train']]
conteggio_classi = Counter(train_labels)

num_classi = len(set(train_labels))

#formula del bilanciamento multi-classe: total_samples / (num_classes * class_samples)
totale_campioni = len(train_labels)
pesi_array = np.zeros(num_classi, dtype=np.float32)

for t in range(num_classi):
        pesi_array[t] = totale_campioni / (num_classi * conteggio_classi[t])


#convertiamo il vettore dei pesi in tensore
ce_weights = torch.tensor(pesi_array, dtype=torch.float32).to(device)

print(f"Task B - Classi rilevate: {num_classi}")

model = GCNClassifier(
    in_channels=dataset['train'][0].num_features, 
    hidden_channels=256, 
    out_channels=num_classi
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
loss_fn = torch.nn.CrossEntropyLoss(weight=ce_weights)
best_model_path = "best_model_task3_b_GCN.pth"
scaler = torch.amp.GradScaler()

# Lancio dell'addestramento
history = train_loop(
    model, 
    train_loader, 
    val_loader, 
    optimizer, 
    loss_fn, 
    device, 
    epochs, 
    best_model_path, 
    scaler,
    patience=5
)

In [ ]:
from utils_task1 import plot_history

plot_history(history,"Training GCN task 3B")

In [ ]:
print("Valutazione sul test set")
model.load_state_dict(torch.load(best_model_path))
test_metrics = evaluate(model,test_loader,loss_fn,device)
print(f"Test Loss: {test_metrics['val_loss']:.4f} - Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f} - F1 Score: {test_metrics['f1_score']:.4f}")
torch.cuda.empty_cache()

Tentativo SAGE

In [ ]:
from utils_task3_b import SAGEClassifier, train_loop, evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lr = 2e-4
epochs = 50

model = SAGEClassifier(
    in_channels=dataset['train'][0].num_features, 
    hidden_channels=256, 
    out_channels=num_classi
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
loss_fn = torch.nn.CrossEntropyLoss(weight=ce_weights)
best_model_path = "best_model_task3_b_SAGE.pth"
scaler = torch.amp.GradScaler()

# Lancio dell'addestramento
history = train_loop(
    model, 
    train_loader, 
    val_loader, 
    optimizer, 
    loss_fn, 
    device, 
    epochs, 
    best_model_path, 
    scaler,
    patience=5
)

In [ ]:
from utils_task1 import plot_history

plot_history(history, "Training SAGE task 3B")

In [ ]:
print("Valutazione sul test set")
model.load_state_dict(torch.load(best_model_path))
test_metrics = evaluate(model, test_loader, loss_fn, device)
print(f"Test Loss: {test_metrics['val_loss']:.4f} - Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f} - F1 Score: {test_metrics['f1_score']:.4f}")
torch.cuda.empty_cache()

Tentativo GAT

In [ ]:
from utils_task3_b import GATv2Classifier, train_loop, evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lr = 2e-4
epochs = 50

model = GATv2Classifier(
    in_channels=dataset['train'][0].num_features, 
    hidden_channels=256, 
    out_channels=num_classi
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
loss_fn = torch.nn.CrossEntropyLoss(weight=ce_weights)
best_model_path = "best_model_task3_b_GAT.pth"
scaler = torch.amp.GradScaler()

# Lancio dell'addestramento
history = train_loop(
    model, 
    train_loader, 
    val_loader, 
    optimizer, 
    loss_fn, 
    device, 
    epochs, 
    best_model_path, 
    scaler,
    patience=5
)

In [ ]:
from utils_task1 import plot_history

plot_history(history, "Training GAT task 3B")

In [ ]:
print("Valutazione sul test set")
model.load_state_dict(torch.load(best_model_path))
test_metrics = evaluate(model, test_loader, loss_fn, device)
print(f"Test Loss: {test_metrics['val_loss']:.4f} - Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f} - F1 Score: {test_metrics['f1_score']:.4f}")
torch.cuda.empty_cache()

Tentativo GIN

In [ ]:
from utils_task3_b import HierarchicalGINClassifier, train_loop, evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lr = 2e-4
epochs = 50

model = HierarchicalGINClassifier(
    in_channels=dataset['train'][0].num_features, 
    hidden_channels=256, 
    out_channels=num_classi
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
loss_fn = torch.nn.CrossEntropyLoss(weight=ce_weights)
best_model_path = "best_model_task3_b_GIN.pth"
scaler = torch.amp.GradScaler()

# Lancio dell'addestramento
history = train_loop(
    model, 
    train_loader, 
    val_loader, 
    optimizer, 
    loss_fn, 
    device, 
    epochs, 
    best_model_path, 
    scaler,
    patience=5
)

In [ ]:
from utils_task1 import plot_history

plot_history(history, "Training GIN task 3B")

In [ ]:
print("Valutazione sul test set")
model.load_state_dict(torch.load(best_model_path))
test_metrics = evaluate(model, test_loader, loss_fn, device)
print(f"Test Loss: {test_metrics['val_loss']:.4f} - Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f} - F1 Score: {test_metrics['f1_score']:.4f}")
torch.cuda.empty_cache()